# Dataset Split

This notebook splits the full dataset into train and test sets with two constraints:

1. **No ARTICLE_ID overlap** between train and test — prevents data leakage at the article level
2. **Balanced RESNET_PRED_CLASS distribution** in the test set — ensures each TEM modality is equally represented

The split is performed at the article level: all subfigures from the same article go to either train or test, never both.

# Import packages

In [ ]:
import os
import csv
import pandas as pd
import numpy as np
from collections import Counter
from pathlib import Path

## Settings

Adjust the paths below to match your local setup:

- `INPUT_CSV_PATH`: full dataset CSV containing `ARTICLE_ID`, `CROP_IMAGE`, and `RESNET_PRED_CLASS` columns
- `OUTPUT_TRAIN_PATH`: output path for the training set
- `OUTPUT_TEST_PATH`: output path for the test set
- `TEST_SIZE`: number of samples for the test set (default: 2564)

In [ ]:
INPUT_CSV_PATH    = "/path/to/your/total_dataset.csv"
OUTPUT_TRAIN_PATH = "/path/to/your/train.csv"
OUTPUT_TEST_PATH  = "/path/to/your/test.csv"

TEST_SIZE = 2564

## Split Function

For each TEM modality class, the function:
1. Groups subfigures by `ARTICLE_ID`
2. Randomly selects complete articles until the per-class target is reached
3. Verifies there is no `ARTICLE_ID` overlap between train and test

In [ ]:
def split_train_test(csv_path, test_size=2564, output_train='train.csv', output_test='test.csv'):
    """
    Split CSV data into train and test sets with the following constraints:
    1. No ARTICLE_ID overlap between train and test
    2. Balanced RESNET_PRED_CLASS distribution in test set
    
    Parameters:
    -----------
    csv_path : str
        Path to input CSV file
    test_size : int
        Number of samples for test set (default: 2564)
    output_train : str
        Output path for training set
    output_test : str
        Output path for test set
    """
    
    print(f"Reading CSV file: {csv_path}")
    df = pd.read_csv(csv_path)
    print(f"Total records: {len(df)}")
    
    if 'ARTICLE_ID' not in df.columns:
        raise ValueError("Column 'ARTICLE_ID' not found in CSV")
    if 'RESNET_PRED_CLASS' not in df.columns:
        raise ValueError("Column 'RESNET_PRED_CLASS' not found in CSV")
    
    class_counts = df['RESNET_PRED_CLASS'].value_counts()
    print(f"\nOriginal class distribution:")
    print(class_counts)
    print(f"\nUnique classes: {len(class_counts)}")

    article_groups = df.groupby('ARTICLE_ID')
    print(f"\nUnique ARTICLE_IDs: {len(article_groups)}")

    article_info = []
    for article_id, group in article_groups:
        dominant_class = group['RESNET_PRED_CLASS'].mode()[0]
        record_count = len(group)
        article_info.append({
            'article_id': article_id,
            'dominant_class': dominant_class,
            'record_count': record_count,
            'indices': group.index.tolist()
        })
    
    article_df = pd.DataFrame(article_info)
    print(f"\nArticle-level statistics:")
    print(f"Total unique articles: {len(article_df)}")
    print(f"Articles per class:")
    print(article_df['dominant_class'].value_counts().sort_index())

    n_classes = len(class_counts)
    samples_per_class = test_size // n_classes
    remainder = test_size % n_classes
    print(f"\nTarget test set size: {test_size}")
    print(f"Base samples per class: {samples_per_class}")
    print(f"Remainder to distribute: {remainder}")

    test_article_ids = set()
    test_indices = []
    
    for idx, class_label in enumerate(sorted(class_counts.index)):
        class_articles = article_df[article_df['dominant_class'] == class_label].copy()
        target = samples_per_class + (1 if idx < remainder else 0)
        class_articles = class_articles.sample(frac=1, random_state=42+idx).reset_index(drop=True)

        current_count = 0
        for _, row in class_articles.iterrows():
            article_id = row['article_id']

            if article_id in test_article_ids:
                continue

            record_count = row['record_count']

            if current_count + record_count <= target * 1.5:  # Allow 50% overshoot
                test_article_ids.add(article_id)
                test_indices.extend(row['indices'])
                current_count += record_count

            if current_count >= target:
                break

        print(f"Class {class_label}: selected {current_count} samples from {len(test_article_ids & set(class_articles['article_id']))} articles (target: {target})")

    
    test_df  = df[df['ARTICLE_ID'].isin(test_article_ids)].copy()
    train_df = df[~df['ARTICLE_ID'].isin(test_article_ids)].copy()

    train_articles = set(train_df['ARTICLE_ID'].unique())
    test_articles  = set(test_df['ARTICLE_ID'].unique())
    overlap = train_articles.intersection(test_articles)

    print(f"\n{'='*50}")
    print(f"VERIFICATION:")
    print(f"{'='*50}")
    print(f"Train set size: {len(train_df)}")
    print(f"Test set size: {len(test_df)}")
    print(f"Unique ARTICLE_IDs in train: {len(train_articles)}")
    print(f"Unique ARTICLE_IDs in test: {len(test_articles)}")
    print(f"ARTICLE_ID overlap: {len(overlap)}")

    if len(overlap) > 0:
        print(f"WARNING: Found {len(overlap)} overlapping ARTICLE_IDs!")
        print(f"Overlapping IDs: {list(overlap)[:10]}...")
    else:
        print("✓ No ARTICLE_ID overlap - data leakage prevented!")

    print(f"\nTest set class distribution:")
    test_class_counts = test_df['RESNET_PRED_CLASS'].value_counts().sort_index()
    print(test_class_counts)
    
    max_count = test_class_counts.max()
    min_count = test_class_counts.min()
    balance_ratio = min_count / max_count if max_count > 0 else 0
    print(f"\nBalance ratio (min/max): {balance_ratio:.2f}")
    print(f"Standard deviation: {test_class_counts.std():.2f}")
    
    train_df.to_csv(output_train, index=False)
    test_df.to_csv(output_test, index=False)
    
    print(f"\n{'='*50}")
    print(f"Files saved:")
    print(f"  Train: {output_train}")
    print(f"  Test: {output_test}")
    print(f"{'='*50}")
    
    return train_df, test_df

## Run Split

In [ ]:
np.random.seed(42)

train_df, test_df = split_train_test(
    csv_path=INPUT_CSV_PATH,
    test_size=TEST_SIZE,
    output_train=OUTPUT_TRAIN_PATH,
    output_test=OUTPUT_TEST_PATH
)